In [28]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# STEP 1: UNDERSTAND YOUR DATA
# ─────────────────────────────────────────────
df = pd.read_csv("vital_signs_node11.csv")  # update path as needed

print("=" * 60)
print("STEP 1: DATA EXPLORATION")
print("=" * 60)
print(f"Shape        : {df.shape}")
print(f"\nColumn names :\n{df.columns.tolist()}")
print(f"Shape        : {df.shape}")


STEP 1: DATA EXPLORATION
Shape        : (40004, 17)

Column names :
['Patient ID', 'Heart Rate', 'Respiratory Rate', 'Timestamp', 'Body Temperature', 'Oxygen Saturation', 'Systolic Blood Pressure', 'Diastolic Blood Pressure', 'Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Derived_HRV', 'Derived_Pulse_Pressure', 'Derived_BMI', 'Derived_MAP', 'Risk Category']
Shape        : (40004, 17)


In [29]:
print(f"\nColumn names :\n{df.columns.tolist()}")
print(f"\nData types   :\n{df.dtypes}")



Column names :
['Patient ID', 'Heart Rate', 'Respiratory Rate', 'Timestamp', 'Body Temperature', 'Oxygen Saturation', 'Systolic Blood Pressure', 'Diastolic Blood Pressure', 'Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Derived_HRV', 'Derived_Pulse_Pressure', 'Derived_BMI', 'Derived_MAP', 'Risk Category']

Data types   :
Patient ID                    int64
Heart Rate                    int64
Respiratory Rate              int64
Timestamp                    object
Body Temperature            float64
Oxygen Saturation           float64
Systolic Blood Pressure       int64
Diastolic Blood Pressure      int64
Age                           int64
Gender                       object
Weight (kg)                 float64
Height (m)                  float64
Derived_HRV                 float64
Derived_Pulse_Pressure        int64
Derived_BMI                 float64
Derived_MAP                 float64
Risk Category                object
dtype: object


In [30]:
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")



Missing values:
Patient ID                  0
Heart Rate                  0
Respiratory Rate            0
Timestamp                   0
Body Temperature            0
Oxygen Saturation           0
Systolic Blood Pressure     0
Diastolic Blood Pressure    0
Age                         0
Gender                      0
Weight (kg)                 0
Height (m)                  0
Derived_HRV                 0
Derived_Pulse_Pressure      0
Derived_BMI                 0
Derived_MAP                 0
Risk Category               0
dtype: int64

Duplicate rows: 0


In [31]:
print(df.describe())

         Patient ID    Heart Rate  Respiratory Rate  Body Temperature  \
count  40004.000000  40004.000000      40004.000000      40004.000000   
mean   20002.500000     79.596540         15.495125         36.746539   
std    11548.304421     11.592337          2.295813          0.433599   
min        1.000000     60.000000         12.000000         36.000004   
25%    10001.750000     70.000000         13.000000         36.369162   
50%    20002.500000     80.000000         15.000000         36.746390   
75%    30003.250000     90.000000         17.000000         37.121302   
max    40004.000000     99.000000         19.000000         37.499976   

       Oxygen Saturation  Systolic Blood Pressure  Diastolic Blood Pressure  \
count       40004.000000             40004.000000              40004.000000   
mean           97.509188               124.427207                 79.505549   
std             1.437492                 8.660635                  5.751970   
min            95.000041  

In [32]:
print("\n" + "=" * 60)
print("STEP 2: DROP IRRELEVANT COLUMNS")
print("=" * 60)

# For ML (Risk Category prediction):
# - Patient ID  : row index, no signal
# - Timestamp   : recording time, no causal link to risk
# - Weight (kg) : redundant — Derived_BMI already encodes this
# - Height (m)  : redundant — Derived_BMI already encodes this
# - Derived_MAP : not selected for this model
columns_to_drop = ["Patient ID", "Timestamp", "Weight (kg)", "Height (m)", "Derived_MAP"]
df.drop(columns=columns_to_drop, inplace=True)

print(f"Dropped columns : {columns_to_drop}")
print(f"Remaining shape : {df.shape}")


STEP 2: DROP IRRELEVANT COLUMNS
Dropped columns : ['Patient ID', 'Timestamp', 'Weight (kg)', 'Height (m)', 'Derived_MAP']
Remaining shape : (40004, 12)


In [33]:
df.head()

,Heart Rate,Respiratory Rate,Body Temperature,Oxygen Saturation,Systolic Blood Pressure,Diastolic Blood Pressure,Age,Gender,Derived_HRV,Derived_Pulse_Pressure,Derived_BMI,Risk Category
0,60,12,36.861707,95.702046,124,86,37,Female,0.121033,38,32.459031,High Risk
1,63,18,36.511633,96.689413,126,84,77,Male,0.117062,42,12.771246,High Risk
2,63,15,37.052049,98.508265,131,78,68,Female,0.053200,53,28.821069,Low Risk
3,99,16,36.654748,95.011801,118,72,41,Female,0.064475,46,28.554611,High Risk
4,69,16,36.975098,98.623792,138,76,25,Female,0.118484,62,16.081438,High Risk


In [34]:
# ─────────────────────────────────────────────
# STEP 3: RENAME COLUMNS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: RENAME COLUMNS")
print("=" * 60)

df.rename(columns={
    "Heart Rate"              : "heart_rate",
    "Respiratory Rate"        : "respiratory_rate",
    "Body Temperature"        : "body_temperature",
    "Oxygen Saturation"       : "oxygen_saturation",
    "Systolic Blood Pressure" : "systolic_bp",
    "Diastolic Blood Pressure": "diastolic_bp",
    "Age"                     : "age",
    "Gender"                  : "gender",
    "Derived_HRV"             : "derived_hrv",
    "Derived_Pulse_Pressure"  : "derived_pulse_pressure",
    "Derived_BMI"             : "derived_bmi",
    "Risk Category"           : "risk_category",
}, inplace=True)


print(df.columns.tolist())


STEP 3: RENAME COLUMNS
['heart_rate', 'respiratory_rate', 'body_temperature', 'oxygen_saturation', 'systolic_bp', 'diastolic_bp', 'age', 'gender', 'derived_hrv', 'derived_pulse_pressure', 'derived_bmi', 'risk_category']


In [35]:
# ─────────────────────────────────────────────
# STEP 4: FIX DATA TYPES
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: FIX DATA TYPES")
print("=" * 60)

# Convert gender and risk_category to category type (saves memory)
df["gender"]        = df["gender"].astype("category")
df["risk_category"] = df["risk_category"].astype("category")

print(df.dtypes)


STEP 4: FIX DATA TYPES
heart_rate                   int64
respiratory_rate             int64
body_temperature           float64
oxygen_saturation          float64
systolic_bp                  int64
diastolic_bp                 int64
age                          int64
gender                    category
derived_hrv                float64
derived_pulse_pressure       int64
derived_bmi                float64
risk_category             category
dtype: object


In [36]:
# STEP 5: REMOVE DUPLICATES
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: REMOVE DUPLICATES")
print("=" * 60)

before = len(df)
df.drop_duplicates(inplace=True)
after  = len(df)

print(f"Rows before : {before}")
print(f"Rows after  : {after}")
print(f"Duplicates removed: {before - after}")


STEP 5: REMOVE DUPLICATES
Rows before : 40004
Rows after  : 40004
Duplicates removed: 0


In [37]:
# STEP 6: HANDLE MISSING VALUES
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: HANDLE MISSING VALUES")
print("=" * 60)

print(f"Missing values per column:\n{df.isnull().sum()}")

# Numeric columns → fill with median (robust to outliers)
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    missing = df[col].isnull().sum()
    if missing > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  Filled '{col}' ({missing} nulls) with median={median_val:.2f}")

# Categorical columns → fill with mode
cat_cols = df.select_dtypes(include=["category", "object"]).columns
for col in cat_cols:
    missing = df[col].isnull().sum()
    if missing > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"  Filled '{col}' ({missing} nulls) with mode='{mode_val}'")

print(f"\nMissing values after imputation:\n{df.isnull().sum()}")


STEP 6: HANDLE MISSING VALUES
Missing values per column:
heart_rate                0
respiratory_rate          0
body_temperature          0
oxygen_saturation         0
systolic_bp               0
diastolic_bp              0
age                       0
gender                    0
derived_hrv               0
derived_pulse_pressure    0
derived_bmi               0
risk_category             0
dtype: int64

Missing values after imputation:
heart_rate                0
respiratory_rate          0
body_temperature          0
oxygen_saturation         0
systolic_bp               0
diastolic_bp              0
age                       0
gender                    0
derived_hrv               0
derived_pulse_pressure    0
derived_bmi               0
risk_category             0
dtype: int64


In [38]:
# ─────────────────────────────────────────────
# STEP 7: FIX STRUCTURAL ERRORS & INCONSISTENCIES
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: FIX STRUCTURAL ERRORS & INCONSISTENCIES")
print("=" * 60)

# Standardize gender values: strip whitespace, title-case
df["gender"] = df["gender"].astype(str).str.strip().str.title().astype("category")
print(f"Gender unique values : {df['gender'].unique().tolist()}")



STEP 7: FIX STRUCTURAL ERRORS & INCONSISTENCIES
Gender unique values : ['Female', 'Male']


In [39]:
# STEP 8: HANDLE OUTLIERS (IQR method)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: HANDLE OUTLIERS")
print("=" * 60)

# Apply IQR capping on key clinical numeric columns
outlier_cols = [
    "heart_rate", "respiratory_rate", "body_temperature",
    "oxygen_saturation", "systolic_bp", "diastolic_bp", "age"
]

for col in outlier_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_count = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)

    print(f"  '{col}': {outlier_count} outliers capped → [{lower:.2f}, {upper:.2f}]")


STEP 8: HANDLE OUTLIERS
  'heart_rate': 0 outliers capped → [40.00, 120.00]
  'respiratory_rate': 0 outliers capped → [7.00, 23.00]
  'body_temperature': 0 outliers capped → [35.24, 38.25]
  'oxygen_saturation': 0 outliers capped → [92.55, 102.49]
  'systolic_bp': 0 outliers capped → [94.50, 154.50]
  'diastolic_bp': 0 outliers capped → [60.00, 100.00]
  'age': 0 outliers capped → [-19.00, 125.00]


In [40]:
# STEP 9: VALIDATE & SANITY CHECK
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: VALIDATE & SANITY CHECK")
print("=" * 60)

checks = {
    "heart_rate (60–100 bpm)"          : df["heart_rate"].between(40, 220),
    "respiratory_rate (10–30)"         : df["respiratory_rate"].between(8, 40),
    "body_temperature (35–42°C)"       : df["body_temperature"].between(34, 43),
    "oxygen_saturation (80–100%)"      : df["oxygen_saturation"].between(80, 100),
    "systolic_bp (80–200 mmHg)"        : df["systolic_bp"].between(70, 250),
    "diastolic_bp (50–130 mmHg)"       : df["diastolic_bp"].between(40, 150),
    "age (0–120)"                      : df["age"].between(0, 120),
    "systolic > diastolic"             : df["systolic_bp"] > df["diastolic_bp"],
}

all_passed = True
for check, mask in checks.items():
    failed = (~mask).sum()
    status = "✅ PASS" if failed == 0 else f"⚠️  FAIL ({failed} rows)"
    print(f"  {status} — {check}")
    if failed > 0:
        all_passed = False

print(f"\nAll checks passed: {all_passed}")
print(f"\nFinal dataset shape: {df.shape}")
print(f"Final dtypes:\n{df.dtypes}")



STEP 9: VALIDATE & SANITY CHECK
  ✅ PASS — heart_rate (60–100 bpm)
  ✅ PASS — respiratory_rate (10–30)
  ✅ PASS — body_temperature (35–42°C)
  ✅ PASS — oxygen_saturation (80–100%)
  ✅ PASS — systolic_bp (80–200 mmHg)
  ✅ PASS — diastolic_bp (50–130 mmHg)
  ✅ PASS — age (0–120)
  ✅ PASS — systolic > diastolic

All checks passed: True

Final dataset shape: (40004, 12)
Final dtypes:
heart_rate                   int64
respiratory_rate             int64
body_temperature           float64
oxygen_saturation          float64
systolic_bp                  int64
diastolic_bp                 int64
age                          int64
gender                    category
derived_hrv                float64
derived_pulse_pressure       int64
derived_bmi                float64
risk_category             category
dtype: object


In [41]:
# ─────────────────────────────────────────────
# STEP 10: SAVE CLEANED DATASET
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 10: SAVE CLEANED DATASET")
print("=" * 60)
df.to_csv("data_cleaned1.csv", index=False)
print(f"   Final shape: {df.shape}")


STEP 10: SAVE CLEANED DATASET
   Final shape: (40004, 12)
